# RACE Dataset — Exploratory Data Analysis
BS (CS) Spring 2026 · AI Lab Project

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Load Data

In [ ]:
train_df = pd.read_csv('../data/raw/train.csv')
val_df   = pd.read_csv('../data/raw/val.csv')
test_df  = pd.read_csv('../data/raw/test.csv')

print('Train:', train_df.shape)
print('Val:  ', val_df.shape)
print('Test: ', test_df.shape)
train_df.head(3)

## 2. Missing Values & Data Types

In [ ]:
print('=== Train null counts ===')
print(train_df.isnull().sum())
print('\n=== dtypes ===')
print(train_df.dtypes)

## 3. Answer Label Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (df, title) in zip(axes, [(train_df,'Train'),(val_df,'Val'),(test_df,'Test')]):
    df['answer'].value_counts().sort_index().plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title(f'{title} — Answer Distribution')
    ax.set_xlabel('Answer Label')
    ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

## 4. Passage Length Distribution

In [ ]:
train_df['article_len'] = train_df['article'].str.split().str.len()
train_df['question_len'] = train_df['question'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train_df['article_len'].hist(bins=50, ax=axes[0], color='teal')
axes[0].set_title('Article Length (words)')
train_df['question_len'].hist(bins=30, ax=axes[1], color='coral')
axes[1].set_title('Question Length (words)')
plt.tight_layout()
plt.show()

print(train_df[['article_len','question_len']].describe())

## 5. Option Length Comparison

In [ ]:
for opt in ['A','B','C','D']:
    train_df[f'{opt}_len'] = train_df[opt].str.split().str.len()

opt_lens = train_df[['A_len','B_len','C_len','D_len']]
opt_lens.plot(kind='box', figsize=(8,4), title='Option Length Distribution')
plt.ylabel('Words')
plt.show()

## 6. Top-20 Most Frequent Words in Questions

In [ ]:
from collections import Counter
import re

all_q_words = ' '.join(train_df['question'].dropna()).lower()
all_q_words = re.sub(r'[^a-z\s]', '', all_q_words)
freq = Counter(all_q_words.split())
common = pd.DataFrame(freq.most_common(20), columns=['word','count'])

common.plot(kind='barh', x='word', y='count', figsize=(8,5),
            title='Top-20 Question Words', legend=False, color='mediumpurple')
plt.xlabel('Frequency')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 7. Summary Statistics Table

In [ ]:
summary = pd.DataFrame({
    'Split':    ['Train', 'Val', 'Test'],
    'Rows':     [len(train_df), len(val_df), len(test_df)],
    'Avg Article Len': [
        train_df['article'].str.split().str.len().mean(),
        val_df['article'].str.split().str.len().mean(),
        test_df['article'].str.split().str.len().mean(),
    ],
    'Avg Question Len': [
        train_df['question'].str.split().str.len().mean(),
        val_df['question'].str.split().str.len().mean(),
        test_df['question'].str.split().str.len().mean(),
    ],
}).round(1)
summary